## <font color=darkcyan> Variational inference </font>
#### <font color=darkorange> Basics: Evidence Lower Bound (ELBO) & Coordinate Ascent Variational Inference (CAVI) 

In [4]:
"""""""""""""""""
Required packages
"""""""""""""""""
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme() 

In [5]:
from numpy.random import default_rng
rng = default_rng()
from scipy import stats
#rng = default_rng(10) # si on veut tous les mêmes simulations

On considère tout d'abord le cas d'étude de l'article : ``Variational Inference: A Review for Statisticians, Blei et al; (2017)``, un mélange de gaussiennes de variance 1. 

#### Mélange de gaussiennes

On considère un mélange de $K$ gaussiennes de moyennes $\mu = (\mu_k)_{1\leqslant k \leqslant K}$ et de variance 1. 

Les variables $\mu = (\mu_k)_{1\leqslant k \leqslant K}$ sont (i.i.d.) de loi  gaussienne de moyenne 0 et de variance $\sigma^2$. 

Les variables $c = (c_i)_{1\leqslant i \leqslant n}$ sont i.i.d., indépendantes de $\mu$ et telles que pour tout $1\leqslant i\leqslant n$, $c_i\in\{1,\ldots,K\}$ suit une loi multinomiale de poids $\{\omega_1,\ldots,\omega_K\}$. 

Conditionnellement à $(\mu,c)$, les observations $(X_i)_{1\leqslant i\leqslant n}$ sont indépendantes et pour tout $1\leqslant i\leqslant n$, la loi conditionnelle de $X_i$ sachant $(\mu,c)$ est gaussienne de moyenne $\mu_{c_i}$ et de variance 1.

Dans la suite, nous notons $\varphi_{\mu_k,\sigma^2}$ la densité gaussienne de moyenne $\mu_k$ et de variance $\sigma^2$. 

#### Question 1
Ecrire une fonction qui simule un échantillon  de taille $1000$ de cette loi avec $K= 3$, $\sigma^2 = 5$, $\omega_k = 1/K$ pour tout $1\leqslant k \leqslant K$.

In [6]:
# Sample data
K  = 3 # number of mixture components
mu = rng.normal(loc=0, scale=np.sqrt(5), size=3) # means of the distribution in each cluster
n_samples = 1000 # number of samples


def ech(size):
    U = stats.randint.rvs(0, K, size=1000)
    S = np.zeros(size)
    N = stats.norm.rvs(loc=mu, scale=np.ones(K), size=(size,K))
    S = N[np.arange(size), U]
    return S

X = ech(1000)
sigma2 = 5

Notre objectif est d'approcher $p(\mu,c|x)$ où $c = (c_1,\cdots,c_n)$ sont les composantes des observations.  En effet $p(\mu,c|x)$ ne peut être calculée explicitement et ne peut être simulée exactement de façon aisée.

L'approximation `mean-field` considérée s'écrit (la dépendance en $x$ est volontairement omise) :

$$
q(\mu,c) = \prod_{k=1}^K \varphi_{m_k,s_k}(\mu_k)\prod_{i=1}^n \mathrm{Cat}_{\phi_i}(c_i)\,, 
$$

ce qui signifie que sous la loi variationelle $q$ :

- $\mu$ et $c$ sont indépendantes.
- $(\mu_{k})_{1\leqslant k \leqslant K}$ sont des gaussiennes indépendantes de moyennes $(m_{k})_{1\leqslant k \leqslant K}$ et variances $(s_{k})_{1\leqslant k \leqslant K}$.
- $(c_{i})_{1\leqslant i \leqslant n}$ sont indépendantes de distribution multinomiales de paramètres $(\phi_i)_{1\leqslant i \leqslant n}$: $q(c_i=k) = \phi_i(k)$ pour $1\leqslant k \leqslant K$. 

Notons $\mathcal{D}$ la famille des distributions telles que $(m_{k})_{1\leqslant k \leqslant K}\in \mathbb{R}^K$,  $(s_{k})_{1\leqslant k \leqslant K}\in (\mathbb{R}_+^*)^K$ et  $(\phi_i)_{1\leqslant i \leqslant n}\in \mathcal{S}_K^n$ où $\mathcal{S}_K$ est le simplexe de dimension $K$. 

L'objectif est de trouver le
"meilleur candidat" dans $\mathcal{D}$ pour approcher $p(\mu,c|x)$, i.e. celui qui minimise ``la distance de Kullback suivante``:

$$
q^* = \mathrm{Argmin}_{q\in\mathcal{D}} \mathrm{KL}\left(q\|p(\cdot|x)\right)\,.
$$

Notez que :
\begin{align*}
\mathrm{KL}\left(q\|p(\cdot|x)\right) &= \mathbb{E}_q[\log q(\mu,c)] - \mathbb{E}_q[\log p(\mu,c|x)]\,,\\
 &= \mathbb{E}_q[\log q(\mu,c)] - \mathbb{E}_q[\log p(\mu,c,x)]+\log p(x)\,,\\
&= -\mathcal{L}_x(q)+\log p(x)\,,
\end{align*}

où l'``Evidence Lower Bound`` (ELBO) est

$$
\mathcal{L}_x(q) = -\mathbb{E}_q[\log q(\mu,c)] + \mathbb{E}_q[\log p(\mu,c,x)] \,.
$$

Ainsi, ``minimiser la divergence de Kullback`` revient à maximiser la ELBO, avec $\log p(x)\geqslant \mathcal{L}_x(q)$.

La complexité de $\mathcal{D}$ détermine la complexité du problème d'optimisation.

### Generic CAVI algorithm

L'algorithme CAVI calcule itérativement pour $1\leqslant k \leqslant K$,

$$
q(\mu_k) \propto \mathrm{exp}\left(\mathbb{E}_{\tilde q_{\mu_k}}[\log p(\mu_k|(\mu_j)_{j\neq k}, c,x)]\right)
$$

et pour tout  $1\leqslant i \leqslant n$,

$$
q(c_i) \propto \mathrm{exp}\left(\mathbb{E}_{\tilde q_{c_i}}[\log p(c_i|(c_\ell)_{\ell \neq i}, \mu, x)]\right)\,,
$$

où $\mathbb{E}_{\tilde q_z}$ est l'espérance sous la loi variationnelle de toutes les variables sauf $z$.

### <font color=darkorange> Application au mélange de lois gaussiennes </font>

### Mise à jour de  $(\phi_i)_{1\leqslant i \leqslant n}$ avec CAVI

#### Question 2
Ecrire la mise à jour explicitement de la loi variationnelle de $c_i$, $1\leqslant i \leqslant n$.

## Mise à jour variationnelle de $q(c_i)$

<div style="font-size: 70%;">

Sous la factorisation mean-field, l'ELBO vue comme fonctionnelle de $q(c_i)$ seule est :

$$
\mathcal{L}_x(q)
=
\sum_k q(c_i=k)\,\tilde{h}(k)
-
\sum_k q(c_i=k)\log q(c_i=k)
+
\text{const}
$$

où

$$
\tilde{h}(k)
=
\mathbb{E}_{q_{-i}}[\log p(\mu,c,x)]
$$

est l'espérance de la log-jointe sur toutes les variables sauf $c_i$.

Nous voulons maximiser $\mathcal{L}_x(q)$ par rapport à $q(c_i)$ sous la contrainte

$$
\sum_k q(c_i=k)=1.
$$

On écrit alors le lagrangien :

$$
\Lambda
=
\sum_k q(c_i=k)\,\tilde{h}(k)
-
\sum_k q(c_i=k)\log q(c_i=k)
-
\lambda\left(\sum_k q(c_i=k) - 1\right)
$$

On dérive par rapport à $q(c_i=k)$ pour chaque $k$ et on égalise à zéro :

$$
\frac{\partial \Lambda}{\partial q(c_i=k)}
=
\tilde{h}(k)
-
\log q(c_i=k)
-
1
-
\lambda
=
0
$$

On isole ensuite $q(c_i=k)$ :

$$
q^*(c_i=k)
=
e^{\tilde{h}(k)-1-\lambda}
=
e^{-1-\lambda}\cdot\exp(\tilde{h}(k))
$$

On impose la contrainte de normalisation :

$$
e^{-1-\lambda}\sum_k\exp(\tilde{h}(k))
=
1
\qquad \Rightarrow \qquad
e^{-1-\lambda}
=
\frac{1}{\sum_{k'}\exp(\tilde{h}(k'))}
$$

En substituant :

$$
q^*(c_i=k)
=
\frac{\exp(\tilde{h}(k))}{\sum_{k'}\exp(\tilde{h}(k'))}
\propto
\exp\left(\mathbb{E}_{q_{-i}}[\log p(\mu,c,x)]\right)
$$
</div>

Si nous nous situons dans le cas concret de notre expérience :

## Mise à jour CAVI pour $q(c_i)$

<div style="font-size: 70%;">

La règle générale de CAVI est :

$$
q(c_i) \propto \exp\left(\mathbb{E}_{q_{-c_i}}[\log p(\mu,c,x)]\right).
$$

Comme $q(c_i)$ est catégorique, on écrit :

$$
\phi_{ik} := q(c_i = k)
\propto
\exp\left(
\mathbb{E}_{q_{-c_i}}[\log p(\mu,c,x)]
\Big|_{c_i = k}
\right).
$$

On décompose ensuite :

$$
\log p(\mu,c,x)
=
\log p(\mu)
+
\log p(c)
+
\log p(x \mid \mu,c).
$$

Les seuls termes qui dépendent de $c_i$ sont :

$$
\log p(c_i)
\quad \text{et} \quad
\log p(x_i \mid \mu, c_i).
$$

Donc :

$$
\mathbb{E}_{q_{-c_i}}[\log p(\mu,c,x)]
\propto
\mathbb{E}_{q_{-c_i}}
\left[
\log p(c_i)
+
\log p(x_i \mid \mu,c_i)
\right].
$$

Pour $c_i = k$ :

$$
p(c_i = k) = \omega_k
\qquad \Rightarrow \qquad
\log p(c_i = k) = \log \omega_k.
$$

De plus,

$$
x_i \mid c_i = k, \mu_k \sim \mathcal{N}(\mu_k,1),
$$

et donc

$$
\log p(x_i \mid c_i = k, \mu_k)
=
-\frac{1}{2}(x_i-\mu_k)^2
-
\frac{1}{2}\log(2\pi).
$$

En prenant l'espérance par rapport à $q(\mu_k)$ avec

$$
\mu_k \sim \mathcal{N}(m_k,s_k),
$$

on obtient :

$$
\mathbb{E}[(x_i-\mu_k)^2]
=
\mathbb{E}[\mu_k^2]
-
2x_i\mathbb{E}[\mu_k]
+
x_i^2.
$$

Comme

$$
\mathbb{E}[\mu_k] = m_k,
\qquad
\mathbb{E}[\mu_k^2] = m_k^2 + s_k,
$$

alors

$$
\mathbb{E}[(x_i-\mu_k)^2]
=
(m_k^2+s_k)-2x_im_k+x_i^2.
$$

Ainsi,

$$
\mathbb{E}_{q(\mu)}[\log p(x_i \mid \mu, c_i=k)]
=
-\frac{1}{2}(m_k^2+s_k)
+
x_im_k
-
\frac{1}{2}x_i^2
-
\frac{1}{2}\log(2\pi).
$$

En ignorant les termes constants qui ne dépendent pas de $k$,

$$
\mathbb{E}_{q(\mu)}[\log p(x_i \mid \mu, c_i=k)]
\propto
-\frac{1}{2}(m_k^2+s_k)
+
x_im_k.
$$

On combine alors les deux contributions :

$$
\mathbb{E}_{q_{-c_i}}[\log p(\mu,c,x)]
\Big|_{c_i=k}
\propto
\log \omega_k
+
x_im_k
-
\frac{1}{2}(m_k^2+s_k).
$$

Finalement,

$$
q(c_i=k)
\propto
\exp\left(
\log \omega_k
+
x_im_k
-
\frac{1}{2}(m_k^2+s_k)
\right).
$$

En utilisant

$$
\exp(\log \omega_k + A_k)
=
\omega_k \exp(A_k),
$$

on obtient :

$$
q(c_i=k)
\propto
\omega_k
\exp\left(
x_im_k
-
\frac{1}{2}(m_k^2+s_k)
\right).
$$

Après normalisation,

$$
\phi_{ik}
=
q(c_i=k)
=
\frac{
\omega_k
\exp\left(
x_im_k
-
\frac{1}{2}(m_k^2+s_k)
\right)
}{
\sum_{h=1}^K
\omega_h
\exp\left(
x_im_h
-
\frac{1}{2}(m_h^2+s_h)
\right)
}.
$$

</div>

#### Question 3
Ecrire une fonction effectuant la mise à jour de la loi variationnelle de $c_i$, $1\leqslant i \leqslant n$.

In [7]:
#on suppose K = 3, et que w_k = 1/K pour tout k
#on suppose aussi que sigma^2 = 5 et que la taille de l'échantillon est de 1000

def CAVI_update_phi(X,m,s2):
    """
    Inputs
    ----------
    X: data
    m: current estimation of the m_k
    s2: current estimation of the s_k
    
    Outputs
    -------
    phi: new estimation of phi
    """
    phi = np.zeros((len(X), K))
    for k in range(K):
        phi[:,k] = 1/3 * np.exp(-0.5*(m[k]**2 + s2[k]-2*m[k]*X))
    phi = phi / np.sum(phi, axis=1)[:,None]
    return phi
    

### Mise à jour de $(m_{k})_{1\leqslant k \leqslant K}$ et $(s_{k})_{1\leqslant k \leqslant K}$ avec CAVI

#### Question 4
Ecrire la mise à jour explicitement pour les $(m_{k})_{1\leqslant k \leqslant K}$ et des $(s_{k})_{1\leqslant k \leqslant K}$.

<div style="font-size: 70%;">

## Mise à jour variationnelle de $q(\mu_k)$

Nous cherchons maintenant la mise à jour de la distribution variationnelle de $\mu_k$ pour $1 \leq k \leq K$.

Sous l'approximation mean-field :

$$
q(\mu_k)
\propto
\exp\left(
\mathbb{E}_{q_{-\mu_k}}[\log p(\mu_k,c,x)]
\right).
$$

Comme seuls le prior sur $\mu_k$ et les termes de vraisemblance associés au cluster $k$ dépendent de $\mu_k$, on a :

$$
p(\mu_k \mid x,c,\mu_{-k})
\propto
p(\mu_k)
\prod_{i=1}^n p(x_i \mid c_i,\mu).
$$

Donc :

$$
\mathbb{E}_{q_{-\mu_k}}[\log p(\mu_k,c,x)]
=
\log p(\mu_k)
+
\sum_{i=1}^n
\mathbb{E}_{q_{-\mu_k}}[\log p(x_i \mid \mu,c_i)].
$$

Le prior sur $\mu_k$ est :

$$
\mu_k \sim \mathcal{N}(0,\sigma^2),
$$

ce qui donne :

$$
\log p(\mu_k)
=
-\frac{\mu_k^2}{2\sigma^2}
+
\text{const}.
$$

Pour la vraisemblance, on utilise :

$$
x_i \mid c_i = k,\mu_k
\sim
\mathcal{N}(\mu_k,1).
$$

Ainsi,

$$
\log p(x_i \mid c_i=k,\mu_k)
=
-\frac{1}{2}(x_i-\mu_k)^2
+
\text{const}.
$$

Comme $q(c_i)$ est catégorique avec paramètres $\phi_i(k)$, on remplace l'indicatrice $1_{c_i=k}$ par son espérance :

$$
\mathbb{E}_{q(c_i)}[1_{c_i=k}]
=
\phi_i(k).
$$

On obtient alors :

$$
\mathbb{E}_{q_{-\mu_k}}[\log p(\mu_k,c,x)]
\propto
-\frac{\mu_k^2}{2\sigma^2}
-
\frac{1}{2}
\sum_{i=1}^n
\phi_i(k)(x_i-\mu_k)^2.
$$

On développe le carré :

$$
(x_i-\mu_k)^2
=
x_i^2
-
2x_i\mu_k
+
\mu_k^2.
$$

Donc :

$$
\mathbb{E}_{q_{-\mu_k}}[\log p(\mu_k,c,x)]
\propto
-\frac{\mu_k^2}{2\sigma^2}
-
\frac{1}{2}
\sum_{i=1}^n
\phi_i(k)
\left(
x_i^2
-
2x_i\mu_k
+
\mu_k^2
\right).
$$

En supprimant les termes constants qui ne dépendent pas de $\mu_k$ :

$$
\mathbb{E}_{q_{-\mu_k}}[\log p(\mu_k,c,x)]
\propto
-\frac{\mu_k^2}{2\sigma^2}
+
\sum_{i=1}^n \phi_i(k)x_i\mu_k
-
\frac{1}{2}
\sum_{i=1}^n \phi_i(k)\mu_k^2.
$$

On regroupe les termes quadratiques et linéaires en $\mu_k$ :

$$
\mathbb{E}_{q_{-\mu_k}}[\log p(\mu_k,c,x)]
\propto
-
\frac{1}{2}
\left(
\frac{1}{\sigma^2}
+
\sum_{i=1}^n \phi_i(k)
\right)
\mu_k^2
+
\left(
\sum_{i=1}^n \phi_i(k)x_i
\right)\mu_k.
$$

Cette expression est la forme canonique d'une loi gaussienne. Ainsi,

$$
q(\mu_k)
=
\mathcal{N}(m_k,s_k),
$$

avec

$$
s_k
=
\frac{1}{
\frac{1}{\sigma^2}
+
\sum_{i=1}^n \phi_i(k)
},
$$

et

$$
m_k
=
\frac{
\sum_{i=1}^n \phi_i(k)x_i
}{
\frac{1}{\sigma^2}
+
\sum_{i=1}^n \phi_i(k)
}.
$$

</div>

#### Question 5
- Ecrire une fonction effectuant la mise à jour des $(m_{k})_{1\leqslant k \leqslant K}$ et des $(s_{k})_{1\leqslant k \leqslant K}$, les autres paramètres étant fixés.

In [8]:
def CAVI_update_mu_s2(X,m,phi,s2,sigma2):
    """
    Inputs
    ----------
    X: data
    m: current estimation of the m_k
    s2: current estimation of the s_k
    phi: current estimation of phi
    sigma2: current estimation of sigma^2
    
    Outputs
    -------
    m, s2: new estimation of the m_k and the s_k
    """
    m = np.zeros(K)
    s2 = np.zeros(K)
    for k in range(K):
        s2[k] = 1/(1/sigma2 + np.sum(phi[:,k]))
        m[k] = s2[k] * np.sum(phi[:,k]*X)
    return m, s2

#### Question 6

- Ecrire explicitement la ELBO(q).
- Ecrire une fonction calculant la ELBO.

In [ ]:

def elbo(X,phi,m,s2,sigma2):
    """
    Inputs
    ----------
    X: data
    m: current estimation of the m_k
    s2: current estimation of the s_k
    phi: current estimation of phi
    sigma2: current estimation of sigma^2
    
    Outputs
    -------
    elbo: value of the elbo with the input parameters
    """
    
    # A compléter

#### Question 7
- Ecrire une fonction effectuant les mises à jour itératives de l'algorithme CAVI.
- Mettre en oeuvre l'algorithme et afficher dans deux graphes séparés la ELBO au fil des itérations et les estimations des paramètres des lois variationnelles au fil des itérations.

In [ ]:
def CAVI_mixture_Gaussian(X,m, s2, phi, sigma2, max_iter = 500, epsilon = 1e-8):
    """
    Inputs
    ----------
    X: data
    m: initial estimation of the m_k
    s2: initial estimation of the s_k
    phi: initial estimation of phi
    sigma2: initial estimation of sigma^2
    
    Outputs
    -------
    elbos: value of the elbo long iterations
    m_est, s2_est: sequence of estimators along iterations
    """
    
    # A compléter

### Sensitibilité aux conditions initiales

#### Question 8
Etudier la sensibilité aux conditions initiales